# SynthID Text: Dual-Key Speculative Decoding with GPT-2

<table class="tfo-notebook-buttons" align="left">
  <td>
    <a target="_blank" href="https://github.com/google-deepmind/synthid-text/blob/main/notebooks/synthid_text_speculative_decoding.ipynb"><img src="https://www.tensorflow.org/images/GitHub-Mark-32px.png" />View source on GitHub</a>
  </td>
</table>

This notebook demonstrates **Dual-Key Tournament Watermarked Speculative
Sampling** with SynthID Text ([Dathathri et al., 2024][synthid-paper]) using two
open-weights GPT-2 models: `gpt2-xl` (48 layers, 1.5B target model) paired with
`distilgpt2` (6 layers, 82M draft model).

1.  **_Setup_**: Loading the target and draft models and configuring
    `DualKeyWatermarkingConfig`, `DualKeySpeculativeEngine`, and
    `DualKeyFrequentistDetector`.
2.  **_Speculative Generation & Token Origins_**: Generating watermarked text
    with speculative decoding, reporting throughput (`tok/s`), draft acceptance
    rate, and tokens per step, and highlighting which tokens were produced by
    the draft model (`key_draft`) versus the target model (`key_target`).
3.  **_Throughput & Watermark Detection_**: Comparing speculative `tok/s`
    against standard autoregressive watermarked generation on `gpt2-xl`, and
    comparing `DualKeyFrequentistDetector` $p$-values on watermarked versus
    unwatermarked completions.

[synthid-paper]: https://doi.org/10.1038/s41586-024-08025-4

# 1. Setup

In [ ]:
# @title Install and import the required Python packages
#
# @markdown Running this cell may require you to restart your session.

! pip install 'synthid-text[notebook]>=0.3.0'

from collections import Counter
import html
import time

from IPython import display
from synthid_text import detector_frequentist
from synthid_text import logits_processing
from synthid_text import speculative_engine
import torch
import transformers

In [ ]:
# @title Load the GPT-2 XL target and DistilGPT-2 draft models

DEVICE = (
    torch.device('cuda:0') if torch.cuda.is_available() else torch.device('cpu')
)
print(f'Running on device: {DEVICE}')

TARGET_MODEL_NAME = 'gpt2-xl'
DRAFT_MODEL_NAME = 'distilgpt2'

tokenizer = transformers.AutoTokenizer.from_pretrained(TARGET_MODEL_NAME)
target_model = (
    transformers.AutoModelForCausalLM.from_pretrained(TARGET_MODEL_NAME)
    .eval()
    .to(DEVICE)
)
draft_model = (
    transformers.AutoModelForCausalLM.from_pretrained(DRAFT_MODEL_NAME)
    .eval()
    .to(DEVICE)
)

In [ ]:
# @title Configure dual-key speculative watermarking and detector

# Keep target_model and draft_model on GPU (DEVICE) while running the small
# (1, top_k) Threefry and tournament watermarking ops on CPU to avoid
# per-round CUDA kernel launch overhead.
CONFIG = speculative_engine.DualKeyWatermarkingConfig(
    ngram_len=5,
    key_draft=42,
    key_target=84,
    depth=15,
    temperature=0.7,
    top_k=40,
    draft_len=4,
    watermark_device=torch.device('cpu'),
    target_device=DEVICE,
    draft_device=DEVICE,
)
EOS_TOKEN_ID = target_model.generation_config.eos_token_id

engine = speculative_engine.DualKeySpeculativeEngine(
    target_model=target_model,
    draft_model=draft_model,
    config=CONFIG,
)
detector = detector_frequentist.DualKeyFrequentistDetector.from_config(
    CONFIG, eos_token_id=EOS_TOKEN_ID
)

# 2. Speculative Generation & Token Origins

During each speculative step, the draft model proposes up to `draft_len`
candidate tokens watermarked with `key_draft`, and the target model verifies
them in a single forward pass. Below, $p_j$ and $q_j$ denote the target and
draft top-k next-token distributions at position $j$. Each generated token in
`output.token_origins` has one of three origins:

*   **`draft`** (green): Proposed by the draft model (watermarked with
    `key_draft`) and accepted by the target model.
*   **`target_residual`** (yellow): Sampled from the residual distribution
    $\max(0, p_j - q_j)$ (watermarked with `key_target`) when a draft candidate
    is rejected.
*   **`target_bonus`** (blue): Sampled from the target model's next-token
    distribution (watermarked with `key_target`) when all `draft_len` draft
    candidates in the step are accepted.

In [ ]:
# @title Helper functions for timing and displaying token origins

_ORIGIN_STYLES = {
    'draft': ('#d4edda', '#155724', 'Draft (key_draft)'),
    'target_residual': ('#fff3cd', '#856404', 'Target residual (key_target)'),
    'target_bonus': ('#cce5ff', '#004085', 'Target bonus (key_target)'),
}


def _sync_device() -> None:
  if torch.cuda.is_available():
    torch.cuda.synchronize()


def display_speculative_output(
    prompt: str,
    output: speculative_engine.SpeculativeGenerationOutput,
    tok_per_sec: float,
    p_value: float,
) -> None:
  """Renders generated tokens color-coded by origin with summary statistics."""
  counts = Counter(origin for origin, _ in output.token_origins)
  tokens_per_step = (
      output.tokens_generated / output.num_steps
      if output.num_steps > 0
      else 0.0
  )

  legend_items = []
  for origin, (bg, fg, label) in _ORIGIN_STYLES.items():
    legend_items.append(
        f'<span style="background-color:{bg}; color:{fg}; padding:2px 6px;'
        ' border-radius:4px; margin-right:8px; font-family:monospace;">'
        f'{html.escape(label)}: {counts[origin]}</span>'
    )

  token_spans = [
      f'<span style="font-weight:bold;">{html.escape(prompt)}</span>'
  ]
  for origin, tok_id in output.token_origins:
    bg, fg, _ = _ORIGIN_STYLES[origin]
    tok_text = html.escape(tokenizer.decode([tok_id])).replace('\n', '↵<br>')
    token_spans.append(
        f'<span style="background-color:{bg}; color:{fg}; padding:1px 2px;'
        ' border-radius:3px; margin:0 1px;"'
        f' title="{origin} (id={tok_id})">{tok_text}</span>'
    )

  block = f"""
  <div style="border:1px solid #ddd; border-radius:6px; padding:12px; margin:10px 0; line-height:1.8;">
    <div style="margin-bottom:8px; font-family:sans-serif; font-size:0.9em;">
      <b>Throughput:</b> {tok_per_sec:.1f} tok/s &nbsp;|&nbsp;
      <b>Acceptance rate:</b> {output.acceptance_rate:.1%}
      ({output.num_accepted_tokens}/{output.num_draft_tokens} draft tokens) &nbsp;|&nbsp;
      <b>Tokens/step:</b> {tokens_per_step:.2f}
      ({output.tokens_generated} tokens in {output.num_steps} steps) &nbsp;|&nbsp;
      <b>Detector p-value:</b> {p_value:.3e}
    </div>
    <div style="margin-bottom:10px; font-size:0.85em;">{''.join(legend_items)}</div>
    <div style="font-family:monospace; white-space:pre-wrap;">{''.join(token_spans)}</div>
  </div>
  """
  display.display(display.HTML(block))

In [ ]:
# @title Generate watermarked sequences and inspect token origins

PROMPTS = [
    'I enjoy walking with my cute dog',
    'The history of artificial intelligence began in',
    'Once upon a time in a small coastal village,',
    'In a distant galaxy, astronomers discovered a',
]
MAX_NEW_TOKENS = 100

# Warm up CUDA kernels before timing.
_warmup_ids = tokenizer(PROMPTS[0], return_tensors='pt').input_ids.to(DEVICE)
_ = engine.generate(_warmup_ids, max_new_tokens=8)
_sync_device()

generator = torch.Generator(device=DEVICE).manual_seed(0)
wm_outputs = []

for prompt in PROMPTS:
  prompt_ids = tokenizer(prompt, return_tensors='pt').input_ids.to(DEVICE)
  _sync_device()
  t0 = time.perf_counter()
  output = engine.generate(
      prompt_ids,
      max_new_tokens=MAX_NEW_TOKENS,
      eos_token_id=EOS_TOKEN_ID,
      generator=generator,
  )
  _sync_device()
  elapsed = time.perf_counter() - t0
  tok_per_sec = output.tokens_generated / elapsed

  generated_ids = output.sequences[:, prompt_ids.shape[1] :]
  p_val = float(detector.p_value(generated_ids)[0])
  wm_outputs.append((prompt, prompt_ids, output, tok_per_sec, p_val))
  display_speculative_output(prompt, output, tok_per_sec, p_val)

# 3. Throughput & Watermark Detection: Speculative vs. Autoregressive

Below we compare:
*   **Throughput (`tok/s`)**: Dual-key speculative decoding (`gpt2-xl` +
    `distilgpt2`) versus standard autoregressive generation on `gpt2-xl`
    with `SynthIDLogitsProcessor` applied after temperature scaling and top-k
    truncation, i.e. to the same candidate distribution the engine watermarks.
*   **Watermark detection (`p_value`)**: `DualKeyFrequentistDetector` pools the
    unmasked tournament $g$-values across both `key_draft` and `key_target` and
    computes an exact one-sided binomial tail probability. Speculatively
    watermarked responses yield small $p$-values, whereas unwatermarked
    responses from `gpt2-xl` follow the Uniform(0, 1) null distribution.

In [ ]:
# @title Compare throughput and detector p-values against autoregressive target-model baselines

torch.manual_seed(0)


def _make_ar_target_processors() -> transformers.LogitsProcessorList:
  """Temperature and top-k first, then SynthID with key_target.

  `generate()` appends its own temperature/top-k warpers *after* any custom
  `logits_processor`, which would run the tournament over the full vocabulary
  on unscaled logits. Building the chain explicitly (and passing
  `temperature=1.0, top_k=0` below so no extra warpers are added) watermarks
  the same top-k candidate distribution that the speculative engine
  watermarks.
  """
  return transformers.LogitsProcessorList([
      transformers.TemperatureLogitsWarper(CONFIG.temperature),
      transformers.TopKLogitsWarper(CONFIG.top_k),
      logits_processing.SynthIDLogitsProcessor(
          ngram_len=CONFIG.ngram_len,
          key=CONFIG.key_target,
          depth=CONFIG.depth,
          context_history_size=CONFIG.context_history_size,
          device=DEVICE,
          skip_first_ngram_calls=CONFIG.skip_first_ngram_calls,
          num_leaves=CONFIG.num_leaves,
      ),
  ])


# Warm up autoregressive generation before timing.
_ = target_model.generate(
    _warmup_ids,
    do_sample=True,
    temperature=1.0,
    top_k=0,
    max_new_tokens=8,
    logits_processor=_make_ar_target_processors(),
    pad_token_id=tokenizer.eos_token_id,
)
_sync_device()

print(
    f"{'Prompt':<38} | {'Accept':>6} | {'Spec tok/s':>10} |"
    f" {'AR WM tok/s':>11} | {'Speedup':>7} | {'Spec WM p':>10} |"
    f" {'Unwm p':>8}"
)
print('-' * 108)

spec_rates = []
ar_rates = []

for prompt, prompt_ids, wm_out, spec_tps, wm_p in wm_outputs:
  # Timed autoregressive watermarked generation on target_model.
  _sync_device()
  t0 = time.perf_counter()
  ar_wm_seq = target_model.generate(
      prompt_ids,
      do_sample=True,
      temperature=1.0,
      top_k=0,
      max_new_tokens=MAX_NEW_TOKENS,
      logits_processor=_make_ar_target_processors(),
      pad_token_id=tokenizer.eos_token_id,
  )
  _sync_device()
  ar_elapsed = time.perf_counter() - t0
  ar_tokens = ar_wm_seq.shape[1] - prompt_ids.shape[1]
  ar_tps = ar_tokens / ar_elapsed
  speedup = spec_tps / ar_tps

  spec_rates.append(spec_tps)
  ar_rates.append(ar_tps)

  # Unwatermarked baseline for detection comparison.
  uwm_seq = target_model.generate(
      prompt_ids,
      do_sample=True,
      temperature=CONFIG.temperature,
      top_k=CONFIG.top_k,
      max_new_tokens=MAX_NEW_TOKENS,
      pad_token_id=tokenizer.eos_token_id,
  )
  uwm_generated_ids = uwm_seq[:, prompt_ids.shape[1] :].to(
      CONFIG.watermark_device
  )
  uwm_p = float(detector.p_value(uwm_generated_ids)[0])

  short_prompt = prompt[:35] + '...' if len(prompt) > 38 else prompt
  print(
      f'{short_prompt:<38} | {wm_out.acceptance_rate:>5.1%} |'
      f' {spec_tps:>10.1f} | {ar_tps:>11.1f} | {speedup:>6.2f}x |'
      f' {wm_p:>10.2e} | {uwm_p:>8.4f}'
  )

mean_spec = sum(spec_rates) / len(spec_rates)
mean_ar = sum(ar_rates) / len(ar_rates)
print('-' * 108)
print(
    f'Mean speculative throughput:     {mean_spec:.1f} tok/s\n'
    f'Mean autoregressive throughput:  {mean_ar:.1f} tok/s\n'
    f'Mean speedup:                    {mean_spec / mean_ar:.2f}x'
)